In [6]:
# 프롬프트 엔지니어링, 파인튜닝의 차이
# zero-shot prompting, few-shot prompting, 전체 매개변수 미세조정(full Fine Turning)
# 성능(accuracy) 자원소모(학습시간, 파라메터 업데이트 비율)
# google/flan-t5-small( seq2seq 소형 모델)

In [7]:
import time
import torch
import pandas as pd
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer,Seq2SeqTrainingArguments,DataCollatorForSeq2Seq
    )
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'using devcies : {device}')

using devcies : cpu


### 데이터셋

In [8]:
train_data = [
    {"text": "This movie was absolutely fantastic! The acting was superb.", "label": "positive"},
    {"text": "I wasted two hours of my life. Terrible plot and cheap CGI.", "label": "negative"},
    {"text": "A true masterpiece of cinema. Highly recommend it to everyone.", "label": "positive"},
    {"text": "Extremely boring and predictable. I fell asleep halfway through.", "label": "negative"},
    {"text": "The cinematography was beautiful, and the story was touching.", "label": "positive"},
    {"text": "Horrible acting and bad direction. Do not watch this film.", "label": "negative"},
    {"text": "Brilliant performance by the lead actor, a must-watch.", "label": "positive"},
    {"text": "The plot made no sense, and the characters were annoying.", "label": "negative"},
    {"text": "I loved the soundtrack and the emotional depth of this movie.", "label": "positive"},
    {"text": "Total waste of money. I regret watching this garbage.", "label": "negative"},
    {"text": "Very engaging storyline with great character development.", "label": "positive"},
    {"text": "Slow, dull, and lacks any real substance.", "label": "negative"},
    {"text": "An amazing adventure that kept me on the edge of my seat.", "label": "positive"},
    {"text": "A complete disappointment. The trailer was much better than the film.", "label": "negative"},
    {"text": "Wonderfully written and beautifully executed. A delight to watch.", "label": "positive"}
]

test_data = [
    {"text": "The movie was mediocre, but the ending was spectacular!", "label": "positive"},
    {"text": "Worst movie I have seen this year. Avoid at all costs.", "label": "negative"},
    {"text": "A refreshing and delightful comedy that had me laughing all night.", "label": "positive"},
    {"text": "The acting was flat and the script felt extremely forced.", "label": "negative"},
    {"text": "An absolute gem of a film with outstanding visuals.", "label": "positive"}
]

print(f"Train dataset size: {len(train_data)}")
print(f"Test dataset size: {len(test_data)}")

Train dataset size: 15
Test dataset size: 5


## 모델 로드 및 모델 매개변수(Parameter) 분석
Hugging Face의 `transformers` 라이브러리를 활용하여 구글이 공개한 지시어 미세조정(Instruction-tuned) 모델인 `google/flan-t5-small`을 로드합니다.
또한 파인튜닝 전에 모델의 **총 매개변수(Total Parameters)** 와 **학습 가능한 매개변수(Trainable Parameters)** 를 확인합니다.

In [9]:
model_name = 'google/flan-t5-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model.to(device)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
              (wo): 

In [16]:
# _, param = next(iter(model.named_parameters()))
# 파라메터 계산함수
def get_trainable_params(model):
    all_param = 0
    trainable_params= 0
    for _, param in model.named_parameters():
        all_param += param.numel()  # 파라메터 개수
        if param.requires_grad:
            trainable_params += param.numel()
    return all_param,trainable_params, trainable_params/all_param*100

all_p, train_p, pct = get_trainable_params(model)
all_p, train_p, pct

(76961152, 76961152, 100.0)